### Prepare Data

In [4]:
#%pip install pandas openpyxl

In [1]:
import pandas as pd

url = "https://raw.githubusercontent.com/aschan12/DADS5250Hackathon/main/DADS5250_Hackathon_Google_Data.xlsx"

# 2. Read directly into a DataFrame
df_salary_threshold = pd.read_excel(url, sheet_name="Agent Salary and Threshold", engine="openpyxl")
df_adv_sign_ups = pd.read_excel(url, sheet_name="Advertiser Sign-ups", engine="openpyxl")
df_exist_staff = pd.read_excel(url, sheet_name="Existing Agent Staffing", engine="openpyxl")

print(df_salary_threshold.head())
print(df_adv_sign_ups.head())
print(df_exist_staff.head())

C:\Users\shkasc\AppData\Local\Temp\ipykernel_2040\852631775.py:1: DeprecationWarning: 
Pyarrow will become a required dependency of pandas in the next major release of pandas (pandas 3.0),
(to allow more performant data types, such as the Arrow string type, and better interoperability with other libraries)
but was not found to be installed on your system.
If this would cause problems for you,
please provide us feedback at https://github.com/pandas-dev/pandas/issues/54466
        
  import pandas as pd


   Country  Annual_Agent_Salary_USD  Advertiser_Eligibility_Threshold_USD
0      USA                    77721                                 35000
1       UK                    51650                                 27000
2  Germany                    51328                                 26000
3    India                    14118                                 13000
4   Brazil                    27950                                 29000
  Advertiser_ID Country         Sign_Up_Date  Projected_Annual_Budget
0       A000001     USA  2024-01-01 00:00:00                    84376
1       A000002     USA  2024-01-01 00:00:00                    57297
2       A000003     USA  2024-01-01 00:00:00                    34081
3       A000004     USA  2024-01-01 00:00:00                   154506
4       A000005     USA  2024-01-01 00:00:00                    23604
   Country  Existing_Agent_Count
0      USA                   652
1       UK                   589
2  Germany                   602
3   

<b> Eligible Ads:  
If Annual Budget > Threshold, then Eligible.  
Remove all Feb 29, 2025 </b>

In [2]:
# Left join the first two tables
eligible_ad = pd.merge(df_adv_sign_ups, df_salary_threshold, on='Country', how='left')
eligible_ad = eligible_ad.drop(columns=['Annual_Agent_Salary_USD'])

# Find eligible advertisers
eligible_ad['Eligible'] = (eligible_ad['Projected_Annual_Budget']> eligible_ad['Advertiser_Eligibility_Threshold_USD'])
eligible_ad = eligible_ad[eligible_ad['Eligible'] == True]

# Drop all on Feb 29, 2025 -> These are the ones that create Error when converting to_datetime
eligible_ad['Sign_Up_Date'] = pd.to_datetime(eligible_ad['Sign_Up_Date'], errors='coerce')
# 2. Drop all rows where 'Sign_Up_Date' became NaT
eligible_ad = eligible_ad.dropna(subset=['Sign_Up_Date'])

eligible_ad.head()

,Advertiser_ID,Country,Sign_Up_Date,Projected_Annual_Budget,Advertiser_Eligibility_Threshold_USD,Eligible
0,A000001,USA,2024-01-01,84376,35000,True
1,A000002,USA,2024-01-01,57297,35000,True
3,A000004,USA,2024-01-01,154506,35000,True
7,A000008,USA,2024-01-01,104967,35000,True
8,A000009,USA,2024-01-01,44165,35000,True


In [179]:
# Past Daily Demands
# daily_demand = eligible_ad.groupby(['Country', 'Sign_Up_Date']).size().reset_index(name='eligible_count')

#daily_demand.head()

In [205]:
# Monthly totals
#eligible_ad['Month'] = (eligible_ad['Sign_Up_Date'].dt.to_period('M'))
#monthly_demand = (eligible_ad.groupby(['Country', 'Month']).size())

#monthly_demand.head()

## Components:
### 1. Simulator by country - then a loop to loop through all countries
#### When an eligible advertiser comes in, they sit in a waiting pool.  
#### Path 1: If there's an agent available, they are assigned to it. Supported for 60 days, and 'graduate'.  
#### Path 2: No available agent, they will continue waiting. Everyday, if there's an agent available, they will be assigned. After 60 days if they are still unassigned, they are removed from the pool.
### 2. Staffing schedule
### 3. Evaluate optimization function

In [3]:
def simulate_country(advertisers, country, staffing_plan, start_date="2024-01-01", end_date="2024-12-31"):
#def simulate_country(advertisers, country, agent_count, start_date="2024-01-01", end_date="2024-12-31"):
    df = advertisers[advertisers['Country'] == country].copy()

    #df["signup_date"] = pd.to_datetime(df["signup_date"])

    df['status'] = 'NOT_ARRIVED'
    df['assignment_date'] = pd.NaT
    df['graduation_date'] = pd.NaT
    df['expiration_date'] = pd.NaT

    daily_results = []

    dates = pd.date_range(start=start_date, end=end_date, freq="D")

    for current_date in dates:

        # Enhancement 1a: dynamic staffing
        current_month = current_date.to_period("M")
        staffing_row = staffing_plan[(staffing_plan["Country"] == country) & (staffing_plan["Month"] == current_month)]
        if staffing_row.empty:
            raise ValueError(f"No staffing data found for " 
                             f"{country}, {current_month}")
        agent_count = int(staffing_row["Agents"].iloc[0])

        # 1. Graduate advertisers
        graduating = ((df["status"] == "ASSIGNED")  & (df["graduation_date"] <= current_date))
        graduated_today = graduating.sum()
        df.loc[graduating, "status"] = "GRADUATED"

        # 2. Add today's advertisers
        arrivals = ((df["Sign_Up_Date"] == current_date) & (df["status"] == "NOT_ARRIVED") )
        new_eligible_today = arrivals.sum()
        df.loc[arrivals, "status"] = "WAITING"

        # 3. Expire old waiting advertisers
        expired = ((df["status"] == "WAITING") & (current_date > df["Sign_Up_Date"]+ pd.Timedelta(days=60)))
        expired_today = expired.sum()
        df.loc[expired, "status"] = "EXPIRED"
        df.loc[expired, "expiration_date"] = current_date

        # 4. Calculate capacity
        total_capacity = agent_count * 10
        currently_supported = (df["status"] == "ASSIGNED").sum()
        available_capacity = max(0, total_capacity - currently_supported)

        # Enhancement 1b: raise error if staff reduction causes currently supported advertisers to suddenly go without an agent
        if currently_supported > total_capacity:
            raise ValueError(f"Infeasible staffing plan: "
                             f"{Country} on {current_date.date()} " f"has {currently_supported} supported advertisers " 
                            f"but only {total_capacity} available capacity.")

        # 5. First-In First-Out assignment
        waiting = (df[df["status"] == "WAITING"].sort_values(["Sign_Up_Date", "Advertiser_ID"]))
        to_assign = waiting.head(available_capacity)
        assigned_today = len(to_assign)
        if assigned_today > 0:
            df.loc[to_assign.index,"status"] = "ASSIGNED"
            df.loc[to_assign.index, "assignment_date"] = current_date
            df.loc[to_assign.index, "graduation_date"] = (current_date + pd.Timedelta(days=60)            )

        # 6. Daily results
        supported = (df["status"] == "ASSIGNED").sum()
        waiting_count = (df["status"] == "WAITING").sum()
        utilization = (supported / total_capacity if total_capacity > 0 else 0)
        daily_results.append({
            "date": current_date,
            "Country": country,
            "agents": agent_count,
            "capacity": total_capacity,
            "new_eligible": new_eligible_today,
            "assigned_today": assigned_today,
            "graduated_today": graduated_today,
            "expired_today": expired_today,
            "supported": supported,
            "waiting": waiting_count,
            "utilization": utilization
        })

    return (df, pd.DataFrame(daily_results))

In [4]:
#staffing_plan = pd.DataFrame({
#    "country": ["USA"] * 12,
#    "month": pd.period_range(
#        "2024-01",
#        "2024-12",
#        freq="M"
#    ),
#    "agents": [
#        50, 50, 55, 55, 60, 60, 60, 60, 58, 58, 56, 60]
#})

#print(staffing_plan)

In [5]:
#us_agents = df_exist_staff.loc[df_exist_staff["Country"] == "USA","Existing_Agent_Count"].iloc[0]

#us_advertisers, us_daily = simulate_country(eligible_ad, country="USA", agent_count=us_agents)
#us_daily.head()

In [6]:
def build_staffing_schedule(initial_agents, decisions):

    decisions = decisions.sort_values("Month").copy()
    active_agents = initial_agents

    previous_hires = 0
    previous_terminations = 0

    results = []

    for _, row in decisions.iterrows():

        # Decisions from previous month
        # take effect this month
        active_agents += previous_hires
        active_agents -= previous_terminations

        if active_agents < 0:
            raise ValueError(f"Active agents became negative " f"in {row['Month']}")

        results.append({
            "Country": row["Country"],
            "Month": row["Month"],
            "Agents": active_agents,
            "Agents_Ramping": int(row["Hires"]),
            "Hires": int(row["Hires"]),
            "Terminations_Notices": int(
                row["Terminations"]
            )
        })

        previous_hires = int(row["Hires"])
        previous_terminations = int(row["Terminations"])

    staffing_plan = pd.DataFrame(results)
    staffing_plan = staffing_plan.merge(df_salary_threshold[["Country", "Annual_Agent_Salary_USD"]],
                                        on="Country", how="left")
    staffing_plan['Monthly_Salary'] = staffing_plan['Annual_Agent_Salary_USD']/12
    staffing_plan["Paid_Agents"] = (staffing_plan["Agents"] + staffing_plan["Agents_Ramping"])
    staffing_plan["Salary_Cost"] = (staffing_plan["Paid_Agents"] * staffing_plan["Monthly_Salary"])
    staffing_plan["Termination_Cost"] = (staffing_plan["Terminations_Notices"] * staffing_plan["Annual_Agent_Salary_USD"]* 0.40)
    staffing_plan["Total_Cost"] = (staffing_plan["Salary_Cost"] + staffing_plan["Termination_Cost"])

    return staffing_plan

In [41]:
def evaluate_plan(advertiser_results, daily_results, staffing_plan):

    completed = advertiser_results[advertiser_results["assignment_date"].notna()].copy()
    completed["wait_days"] = (completed["assignment_date"] - completed["Sign_Up_Date"]).dt.days
    max_wait = completed["wait_days"].max()
    avg_wait = completed["wait_days"].mean()
    p90_wait = completed["wait_days"].quantile(0.90)
    expired = (advertiser_results["status"] == "EXPIRED").sum()
    expiration_rate = (expired / len(advertiser_results))
    avg_utilization = (daily_results["utilization"].mean())
    peak_utilization = (daily_results["utilization"].max())

    return {
        "advertisers_supported": len(completed),
        "advertisers_expired": int(expired),
        "average_wait_days": avg_wait,
        "p90_wait_days": p90_wait,
        "average_utilization": avg_utilization,
        "expired_advertisers": expired,
        "expiration_rate": expiration_rate,
        "total_staffing_cost": staffing_plan["Total_Cost"].sum()
    }
    

### Now that we have the three functions, we can start building for 2026.
#### The goal is to predict staffing for each country for 2026.
#### First, forecast 2026 demand based on 2024-2025 data. Since the functions need both advertiser and monthly counts, we need to make two forecasts.  
#####  One for monthly, another for advertiser-level forecast.
#### Next, make two Staffing schedules: one using no change (baseline), second using GenAI
#### Put both through simulator and evaluate
#### Compare both performance

In [29]:
import numpy as np
import pandas as pd

# Monthly totals
eligible_ad['Month'] = (eligible_ad['Sign_Up_Date'].dt.to_period('M'))
monthly_demand = (eligible_ad.groupby(['Country', 'Month']).size().reset_index(name="Eligible_Advertisers"))
monthly_demand["Year"] = (monthly_demand["Month"].dt.year)
monthly_demand["Month_Number"] = (monthly_demand["Month"].dt.month)

demand_history = (monthly_demand.pivot_table(index=["Country", "Month_Number"],
        columns="Year",
        values="Eligible_Advertisers",
        aggfunc="sum",
        fill_value=0).reset_index())
demand_history = demand_history.rename(columns={2024: "Demand_2024", 2025: "Demand_2025"})

demand_history["Growth_Rate"] = ((demand_history["Demand_2025"] - demand_history["Demand_2024"])/ demand_history["Demand_2024"])
demand_history.head()


Year,Country,Month_Number,Demand_2024,Demand_2025,Growth_Rate
0,Argentina,1,53,90,0.698113
1,Argentina,2,66,70,0.060606
2,Argentina,3,69,59,-0.144928
3,Argentina,4,71,79,0.112676
4,Argentina,5,67,65,-0.029851


In [33]:
import pandas as pd
import numpy as np

def forecast_2026_monthly_demand(monthly_demand, min_growth=-0.20, max_growth=0.30):
    """
        Forecasts monthly eligible advertiser demand for 2026 using 2024 and 2025 monthly demand.
    
        Forecast logic:
            Growth = (2025 Demand - 2024 Demand) / 2024 Demand
    
            2026 Forecast = 2025 Demand * (1 + Applied Growth)

        Growth is capped to prevent extreme forecasts.
        min_growth : float
            Minimum allowed growth rate.
            Default = -20%.

        max_growth : float
            Maximum allowed growth rate.
            Default = +30%.
    
    """

    # Cap extreme growth
    monthly_demand["Applied_Growth"] = (monthly_demand["Growth_Rate"].clip(lower=min_growth, upper=max_growth))

    # Forecast 2026
    monthly_demand["Forecast_2026"] = (monthly_demand["Demand_2025"] * (1 + monthly_demand["Applied_Growth"])).round().astype(int)

    # Create 2026 Month
    monthly_demand["Month"] = pd.PeriodIndex([f"2026-{month:02d}"for month in monthly_demand["Month_Number"]],freq="M")

    # Final output
    forecast = monthly_demand[
        [
            "Country",
            "Month",
            "Month_Number",
            "Demand_2024",
            "Demand_2025",
            "Growth_Rate",
            "Applied_Growth",
            "Forecast_2026"
        ]
    ].copy()


    return (forecast.sort_values(["Country", "Month"]).reset_index(drop=True))

In [35]:
forecast_2026 = forecast_2026_monthly_demand(demand_history)
forecast_2026.head()

Year,Country,Month,Month_Number,Demand_2024,Demand_2025,Growth_Rate,Applied_Growth,Forecast_2026
0,Argentina,2026-01,1,53,90,0.698113,0.300000,117
1,Argentina,2026-02,2,66,70,0.060606,0.060606,74
2,Argentina,2026-03,3,69,59,-0.144928,-0.144928,50
3,Argentina,2026-04,4,71,79,0.112676,0.112676,88
4,Argentina,2026-05,5,67,65,-0.029851,-0.029851,63


In [37]:
def create_2026_forecast_advertisers(forecast_2026, historical_advertisers, random_seed=42):
    """
    Converts monthly 2026 demand forecasts into advertiser-level
    records for the daily staffing simulator.
    
    historical_advertisers : from 2024-2025
    random_seed : int
        Makes the generated forecast reproducible.

    """

    forecast = forecast_2026.copy()
    historical = historical_advertisers.copy()

    # Ensure dates have the correct type
    historical["Sign_Up_Date"] = pd.to_datetime(historical["Sign_Up_Date"])

    # Keep only the required historical years
    historical = historical[historical["Sign_Up_Date"].dt.year.isin([2024, 2025])].copy()

    historical["Month_Number"] = (historical["Sign_Up_Date"].dt.month)

    historical["Day_Of_Month"] = (historical["Sign_Up_Date"].dt.day)

    rng = np.random.default_rng(random_seed)

    generated_rows = []
    advertiser_number = 1

    # Generate advertisers for each Country + Month
    for _, row in forecast.iterrows():

        country = row["Country"]
        month_number = int(row["Month_Number"])

        forecast_count = int(row["Forecast_2026"])

        # Historical advertisers from the same country and calendar month
        historical_pool = historical[(historical["Country"] == country) & (historical["Month_Number"] == month_number)]

        # Sample historical advertisers
        sampled = historical_pool.sample(n=forecast_count, replace=True, random_state=(random_seed + advertiser_number))
        sampled = sampled.reset_index(drop=True)

        # Create corresponding 2026 records
        for _, hist_row in sampled.iterrows():

            signup_date = pd.Timestamp(year=2026, month=month_number, day=int(hist_row["Day_Of_Month"]))

            generated_rows.append({
                "Advertiser_ID": f"F2026_{advertiser_number:08d}",
                "Country": country,
                "Sign_Up_Date": signup_date,
                "Projected_Annual_Budget": hist_row["Projected_Annual_Budget"],
                "Data_Type":"2026_FORECAST"})

            advertiser_number += 1

    forecast_advertisers = pd.DataFrame(generated_rows)

    return (forecast_advertisers.sort_values(["Country", "Sign_Up_Date"]).reset_index(drop=True))

In [39]:
ad_forecast_2026 = create_2026_forecast_advertisers(forecast_2026, historical_advertisers=eligible_ad)
ad_forecast_2026.head()

,Advertiser_ID,Country,Sign_Up_Date,Projected_Annual_Budget,Data_Type
0,F2026_00000036,Argentina,2026-01-02,7026,2026_FORECAST
1,F2026_00000072,Argentina,2026-01-02,8930,2026_FORECAST
2,F2026_00000077,Argentina,2026-01-02,28609,2026_FORECAST
3,F2026_00000103,Argentina,2026-01-02,8930,2026_FORECAST
4,F2026_00000005,Argentina,2026-01-03,9422,2026_FORECAST


### Find Baseline For All Countries in 2026
<b> Baseline = Agent count stays the same </b>

In [48]:
all_staffing_plans = []
all_advertiser_results = []
all_daily_results = []
all_baseline_summary = []

for _, row in df_exist_staff.iterrows():

    country = row["Country"]
    agents = int(row["Existing_Agent_Count"])
    baseline_decisions = pd.DataFrame({
        "Country": [country] * 12,
        "Month": pd.period_range(
            "2026-01",
            "2026-12",
            freq="M"),
        "Hires": [0] * 12,
        "Terminations": [0] * 12
    })

    baseline_staffing_plan = build_staffing_schedule(initial_agents=agents, decisions=baseline_decisions)
    advertiser_result, daily_result = simulate_country(ad_forecast_2026, country=country, staffing_plan=baseline_staffing_plan, start_date="2026-01-01", end_date="2026-12-31")

    all_staffing_plans.append(baseline_staffing_plan)
    all_advertiser_results.append(advertiser_result)
    all_daily_results.append(daily_result)

    #evaluate_plan(advertiser_results= advertiser_result,daily_results = daily_result, staffing_plan = baseline_staffing_plan)
    

simulation_daily = pd.concat(all_daily_results, ignore_index=True)
simulation_advertisers = pd.concat(all_advertiser_results,ignore_index=True) 
simulation_staffing = pd.concat(all_staffing_plans, ignore_index=True)

In [50]:
simulation_daily.head()

,date,Country,agents,capacity,new_eligible,assigned_today,graduated_today,expired_today,supported,waiting,utilization
0,2026-01-01,USA,652,6520,64,64,0,0,64,0,0.009816
1,2026-01-02,USA,652,6520,37,37,0,0,101,0,0.015491
2,2026-01-03,USA,652,6520,71,71,0,0,172,0,0.026380
3,2026-01-04,USA,652,6520,67,67,0,0,239,0,0.036656
4,2026-01-05,USA,652,6520,28,28,0,0,267,0,0.040951


In [51]:
simulation_advertisers.head()

,Advertiser_ID,Country,Sign_Up_Date,Projected_Annual_Budget,Data_Type,status,assignment_date,graduation_date,expiration_date
0,F2026_00088289,USA,2026-01-01,250000,2026_FORECAST,GRADUATED,2026-01-01,2026-03-02,NaT
1,F2026_00088295,USA,2026-01-01,194245,2026_FORECAST,GRADUATED,2026-01-01,2026-03-02,NaT
2,F2026_00088309,USA,2026-01-01,53889,2026_FORECAST,GRADUATED,2026-01-01,2026-03-02,NaT
3,F2026_00088319,USA,2026-01-01,111326,2026_FORECAST,GRADUATED,2026-01-01,2026-03-02,NaT
4,F2026_00088347,USA,2026-01-01,194245,2026_FORECAST,GRADUATED,2026-01-01,2026-03-02,NaT


In [53]:
#simulation_staffing.head()

In [49]:
#evaluate_plan(advertiser_results= us_baseline_ad,daily_results = us_baseline_daily, staffing_plan = baseline_staffing_plan )

In [54]:
def evaluate_country(country, simulation_advertisers, simulation_daily, simulation_staffing):

    advertiser_results = simulation_advertisers[simulation_advertisers["Country"] == country]
    daily_results = simulation_daily[simulation_daily["Country"] == country]
    staffing_plan = simulation_staffing[simulation_staffing["Country"] == country]
    metrics = evaluate_plan(advertiser_results=advertiser_results,daily_results=daily_results,staffing_plan=staffing_plan)
    metrics["Country"] = country

    return metrics

In [55]:
argentina_metrics = evaluate_country("Argentina", simulation_advertisers, simulation_daily, simulation_staffing)

argentina_metrics

{'advertisers_supported': 893,
 'advertisers_expired': 0,
 'average_wait_days': 0.0,
 'p90_wait_days': 0.0,
 'average_utilization': 0.06590136986301369,
 'expired_advertisers': 0,
 'expiration_rate': 0.0,
 'total_staffing_cost': 3737400.0,
 'Country': 'Argentina'}